In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/babies-food-ingredients/food_Ingredients.csv


In [2]:
df = pd.read_csv('/kaggle/input/babies-food-ingredients/food_Ingredients.csv')

pd.set_option('display.max_columns',400)
df.head()

,Unnamed: 0,size,calories_kcal,fats_g,sod_mg,carb_g,fiber_g,sugar_g,protein_g,vitA_g,calcium_mg,thiamin_mg,zinc_mg,potassium_mg,magnesium_mg,vitE_mg,vitK_mcg,vitC_mg,vitB6_mg,copper_mg,carotene_mg,carotene_mcg,cryptoxanthin_mcg,lycopene_mcg,cholesterol_mg,quality
0,0,1.0,21.26,0.20,8.79,4.45,0.00,2.69,0.37,0.00,1.70,0.07,2.13,13.61,3.12,0.03,0.17,6.95,0.07,0.02,3.40,1.98,1.13,0.0,0.00,1
1,1,NaN,82.49,0.79,2.26,17.40,0.90,11.85,1.47,33.90,10.17,0.47,0.40,53.11,12.43,0.18,2.49,24.63,0.23,0.08,14.69,2.26,9.04,0.0,0.00,0
2,2,1.0,20.70,0.20,0.57,4.37,0.23,2.97,0.37,8.51,2.55,0.12,0.10,13.32,3.12,0.05,0.62,6.18,0.06,0.02,3.69,0.57,2.27,0.0,0.00,1
3,3,1.0,32.89,1.16,13.04,4.34,0.31,0.00,1.42,29.77,62.37,0.14,0.26,57.83,9.92,0.00,0.00,0.37,0.02,0.03,0.00,0.00,0.00,0.0,3.12,3
4,4,1.0,58.95,0.90,17.85,11.01,0.78,2.06,1.80,10.35,97.65,0.54,0.28,109.65,17.70,0.75,0.45,0.70,0.06,0.06,4.20,4.05,0.00,0.0,0.00,1


In [3]:

from sklearn.impute import KNNImputer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
import xgboost as xg

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [4]:
df.shape

(696, 26)

In [5]:
df.isnull().sum()

Unnamed: 0             0
size                 162
calories_kcal          0
fats_g                 0
sod_mg                 0
carb_g                 0
fiber_g                0
sugar_g                0
protein_g              0
vitA_g                 0
calcium_mg             0
thiamin_mg             0
zinc_mg                0
potassium_mg           0
magnesium_mg           0
vitE_mg                0
vitK_mcg               0
vitC_mg                0
vitB6_mg               0
copper_mg              0
carotene_mg            0
carotene_mcg           0
cryptoxanthin_mcg      0
lycopene_mcg           0
cholesterol_mg         0
quality                0
dtype: int64

In [6]:
df['size'] = df['size'].fillna(df['size'].mean())

In [7]:
df.corr()

,Unnamed: 0,size,calories_kcal,fats_g,sod_mg,carb_g,fiber_g,sugar_g,protein_g,vitA_g,calcium_mg,thiamin_mg,zinc_mg,potassium_mg,magnesium_mg,vitE_mg,vitK_mcg,vitC_mg,vitB6_mg,copper_mg,carotene_mg,carotene_mcg,cryptoxanthin_mcg,lycopene_mcg,cholesterol_mg,quality
Unnamed: 0,1.000000,0.024037,-0.130624,-0.069390,-0.078261,-0.132156,0.154361,-0.075215,0.105595,0.205272,-0.059012,-0.214159,0.034564,0.313483,0.185539,0.093076,0.129426,0.123797,-0.004383,0.051455,0.234249,0.082685,0.020279,-0.147889,-0.106916,0.024099
size,0.024037,1.000000,0.290402,0.072471,0.078163,0.290080,0.223362,0.197484,0.073272,0.070431,0.133750,0.106182,0.160386,0.202058,0.176741,0.198672,0.010908,0.106864,0.110287,0.177938,0.071244,0.059522,0.080270,0.075412,0.048934,-0.230316
calories_kcal,-0.130624,0.290402,1.000000,0.549446,0.441335,0.785691,0.429748,0.486873,0.502743,0.201098,0.422707,0.296061,0.489110,0.590389,0.478413,0.358780,0.055956,0.144035,0.441927,0.487693,0.194683,0.141651,0.186162,0.218350,0.413835,-0.447876
fats_g,-0.069390,0.072471,0.549446,1.000000,0.569106,-0.057810,0.021728,-0.195280,0.826814,0.094346,0.298666,0.135896,0.725327,0.189963,0.273895,0.122243,0.068092,-0.237590,0.267634,0.121671,0.071531,0.135433,-0.124241,0.238641,0.666429,0.185394
sod_mg,-0.078261,0.078163,0.441335,0.569106,1.000000,0.101123,0.045425,-0.115092,0.543353,0.198012,0.224536,0.109705,0.480502,0.254400,0.271924,0.088344,0.051365,-0.146453,0.192368,0.116424,0.162550,0.194849,-0.070491,0.372179,0.354389,0.071527
carb_g,-0.132156,0.290080,0.785691,-0.057810,0.101123,1.000000,0.528601,0.752657,-0.088184,0.159356,0.281686,0.225188,-0.035309,0.533576,0.349783,0.352601,0.028897,0.361434,0.297697,0.494570,0.166823,0.069472,0.324693,0.074777,-0.023272,-0.694512
fiber_g,0.154361,0.223362,0.429748,0.021728,0.045425,0.528601,1.000000,0.290071,0.033805,0.396173,0.253823,0.116466,0.083184,0.660068,0.585441,0.510476,0.233554,0.066567,0.211737,0.521579,0.404278,0.334030,0.055644,0.141978,-0.107831,-0.489003
sugar_g,-0.075215,0.197484,0.486873,-0.195280,-0.115092,0.752657,0.290071,1.000000,-0.232040,-0.017960,0.112865,0.011375,-0.194758,0.311190,0.065494,0.258733,0.003642,0.461742,0.118498,0.340584,-0.001197,-0.053147,0.393515,-0.087233,-0.035494,-0.782193
protein_g,0.105595,0.073272,0.502743,0.826814,0.543353,-0.088184,0.033805,-0.232040,1.000000,0.099267,0.277525,0.155865,0.899776,0.300492,0.359087,0.135203,0.102303,-0.244871,0.319869,0.132594,0.085255,0.100496,-0.129970,0.223136,0.656959,0.220396
vitA_g,0.205272,0.070431,0.201098,0.094346,0.198012,0.159356,0.396173,-0.017960,0.099267,1.000000,0.246753,0.077100,0.146396,0.624070,0.413175,0.390781,0.228002,-0.067173,0.249864,0.278760,0.990439,0.871518,-0.046064,0.136147,-0.024261,-0.192375


In [8]:
my_pipeline = Pipeline(steps=[
    ('model',xg.XGBRegressor())
])

In [9]:
X = df.drop(['quality'],axis=1)
y= df['quality']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=10)

In [10]:
my_pipeline.fit(X_train,y_train)

Pipeline(steps=[('model',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=None, early_stopping_rounds=None,
                              enable_categorical=False, eval_metric=None,
                              feature_types=None, gamma=None, gpu_id=None,
                              grow_policy=None, importance_type=None,
                              interaction_constraints=None, learning_rate=None,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, n_estimators=100,
                              n_jobs=None, num_parallel_tree=None,
                              predictor=None, random_state=None, ...))])

In [11]:
my_pipeline.score(X_test,y_test)

0.9795630573880234

Doing some feature selection

In [12]:
X = df.drop(['quality'],axis=1)
y = df['quality']
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=10)
from sklearn.feature_selection import SelectKBest, f_classif

np.seterr(divide='ignore', invalid='ignore')
# Select top 10 features using ANOVA
t = SelectKBest(f_classif, k=15).fit(X_train, y_train)
st = X.columns[t.get_support()].tolist()
print(st)

['size', 'calories_kcal', 'carb_g', 'fiber_g', 'sugar_g', 'protein_g', 'potassium_mg', 'magnesium_mg', 'vitE_mg', 'vitC_mg', 'vitB6_mg', 'copper_mg', 'carotene_mg', 'cryptoxanthin_mcg', 'lycopene_mcg']


In [13]:
feat = ['size', 'calories_kcal', 'carb_g', 'fiber_g', 'sugar_g', 'protein_g', 'potassium_mg', 'magnesium_mg', 'vitE_mg', 'vitC_mg', 'vitB6_mg', 'copper_mg', 'carotene_mg', 'cryptoxanthin_mcg', 'lycopene_mcg']
X = df[feat]
y= df['quality']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=10)

In [14]:
my_pipeline.fit(X_train,y_train)
my_pipeline.score(X_test,y_test)

0.9768685074703096